# Time Series Analysis Exercises

## Exercise 0: Environment and libraries
The goal of this exercise is to set up the Python work environment with the required libraries.

- Python >= 3.9
- pandas
- numpy
- jupyter
- plotly

We assume these libraries are installed in the current environment.

In [31]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

Pandas version: 2.0.3
Numpy version: 1.24.3


## Exercise 1: Series
The goal of this exercise is to learn to manipulate time series in Pandas.

1. Create a Series named `integer_series` from 1st January 2010 to 31 December 2020. At each date is associated the number of days since 1st January 2010. It starts with 0.
2. Using Pandas, compute a 7 days moving average. This transformation smooths the time series by removing small fluctuations. **without for loop**.

In [32]:
# 1. Create the Series
dates = pd.date_range(start='2010-01-01', end='2020-12-31', freq='D')
values = np.arange(len(dates))
integer_series = pd.Series(data=values, index=dates, name='integer_series')

print(integer_series.head())
print(integer_series.tail())

2010-01-01    0
2010-01-02    1
2010-01-03    2
2010-01-04    3
2010-01-05    4
Freq: D, Name: integer_series, dtype: int64
2020-12-27    4013
2020-12-28    4014
2020-12-29    4015
2020-12-30    4016
2020-12-31    4017
Freq: D, Name: integer_series, dtype: int64


In [33]:
# 2. Compute 7-day moving average
moving_average = integer_series.rolling(window=7).mean()

print(moving_average.head(10))

2010-01-01    NaN
2010-01-02    NaN
2010-01-03    NaN
2010-01-04    NaN
2010-01-05    NaN
2010-01-06    NaN
2010-01-07    3.0
2010-01-08    4.0
2010-01-09    5.0
2010-01-10    6.0
Freq: D, Name: integer_series, dtype: float64


## Exercise 2: Financial data
This exercise aims to familiarize you with handling financial data using Pandas.

The data we will use is Apple stock.

1. Preprocess: check for missing values, convert string dates to datetime objects, and set the date column as the index.
2. Use Plotly to generate a candlestick chart (Open, High, Low, Close).
3. Aggregate the data to last business day of each month.
4. Compute daily returns based on the Open price without using a for loop.

In [34]:
# Load data
try:
    df = pd.read_csv('../data/AAPL.csv')
except FileNotFoundError:
    # Fallback if running from notebooks dir directly or root
    try:
        df = pd.read_csv('data/AAPL.csv')
    except FileNotFoundError:
        print("Data file not found. Please run scripts/download_data.py first.")

# 1. Preprocess
# Check for missing values
print("Missing values:\n", df.isnull().sum())

# Convert Date to datetime
df['Date'] = pd.to_datetime(df['Date'])

# Set Date as index
df.set_index('Date', inplace=True)

# Sort index just in case
df.sort_index(inplace=True)

print(df.head())

Missing values:
 Date         0
Open         1
High         1
Low          1
Close        1
Adj Close    1
Volume       1
dtype: int64
                Open      High       Low     Close  Adj Close       Volume
Date                                                                      
1980-12-12  0.128348  0.128906  0.128348  0.128348   0.101073  469033600.0
1980-12-15  0.122210  0.122210  0.121652  0.121652   0.095800  175884800.0
1980-12-16  0.113281  0.113281  0.112723  0.112723   0.088768  105728000.0
1980-12-17  0.115513  0.116071  0.115513  0.115513   0.090965   86441600.0
1980-12-18  0.118862  0.119420  0.118862  0.118862   0.093603   73449600.0


In [35]:
# 2. Plotly Candlestick Chart
fig = go.Figure(data=[go.Candlestick(x=df.index,
                open=df['Open'],
                high=df['High'],
                low=df['Low'],
                close=df['Close'])])

fig.update_layout(title='Apple Stock Price', yaxis_title='Price')
fig.show()

In [36]:
# 3. Aggregate to last business day of each month
# Audit requirement: Prices -> mean, Low -> min, High -> max, Volume -> sum
agg_dict = {
    'Open': 'mean',
    'High': 'max',
    'Low': 'min',
    'Close': 'mean',
    'Adj Close': 'mean',
    'Volume': 'sum'
}

monthly_df = df.resample('BM').agg(agg_dict)

print("Number of months:", len(monthly_df))
print(monthly_df.head())

Number of months: 482
                Open      High       Low     Close  Adj Close        Volume
Date                                                                       
1980-12-31  0.136075  0.161272  0.112723  0.135903   0.107022  1.344851e+09
1981-01-30  0.141768  0.155134  0.126116  0.141316   0.111285  6.089888e+08
1981-02-27  0.118215  0.128906  0.106027  0.117892   0.092839  3.216192e+08
1981-03-31  0.111328  0.120536  0.096540  0.110871   0.087310  7.007168e+08
1981-04-30  0.121811  0.131138  0.108259  0.121545   0.095716  5.369280e+08


In [37]:
# 4. Compute daily returns based on Open price
# Method 1: pct_change
df['Open_Return'] = df['Open'].pct_change()

print(df[['Open', 'Open_Return']].head())

                Open  Open_Return
Date                             
1980-12-12  0.128348          NaN
1980-12-15  0.122210    -0.047823
1980-12-16  0.113281    -0.073063
1980-12-17  0.115513     0.019703
1980-12-18  0.118862     0.028992


## Exercise 3: Multi asset returns
The goal of this exercise is to learn to compute daily returns on a DataFrame that contains many assets.

Without using a for loop, compute the daily returns for all the companies and returns a DataFrame.

In [38]:
business_dates = pd.bdate_range('2021-01-01', '2021-12-31')

# Generate tickers
tickers = ['AAPL', 'FB', 'GE', 'AMZN', 'DAI']

# Create index
index = pd.MultiIndex.from_product([business_dates, tickers], names=['Date', 'Ticker'])

# Create DF
market_data = pd.DataFrame(index=index,
                        data=np.random.randn(len(index), 1),
                        columns=['Price'])

# Use pivot_table as requested by audit
pivoted_data = market_data.pivot_table(values="Price", index="Date", columns="Ticker")

# Compute daily returns
returns_df = pivoted_data.pct_change()

print(returns_df.head())

Ticker          AAPL      AMZN       DAI        FB        GE
Date                                                        
2021-01-01       NaN       NaN       NaN       NaN       NaN
2021-01-04 -1.535579 -1.357606  0.105773 -1.172642 -1.658726
2021-01-05 -2.458998  3.552715  2.358122  8.086789  0.126539
2021-01-06  1.226746 -2.433319 -0.902547 -0.441788 -2.397138
2021-01-07 -0.877847 -0.061396 -0.822092 -1.723771 -1.524890


## Exercise 4: Backtest
We will backtest a long only strategy on Apple Inc. 

1. Drop rows with missing values and compute daily future return on Adjusted Close price.
2. Create a Series with random boolean array (p=0.5) for signals.
3. Backtest the signal: Buy 1$ if Signal=1 at d, sell at d+1. Profit = PnL / Invested.
   - PnL for day d: affected to day d.
4. Compute strategy return: (Total Earned - Total Invested) / Total Invested.
5. Compute "Always Buy" strategy PnL and total PnL.
6. Plot daily PnL of both.

In [39]:
# 1. Drop missing and compute future return
# Audit instruction implies: compute return then use it. Or using full index for signals.
# Let's keep the full DF structure to maintain index integrity for signals

df_backtest = df.copy()

# Future return: Return(t) = (Price(t+1) - Price(t)) / Price(t)
df_backtest['daily_return'] = df_backtest['Adj Close'].pct_change()
df_backtest['future_return'] = (df_backtest['Adj Close'].shift(-1) - df_backtest['Adj Close']) / df_backtest['Adj Close']

# Valid rows for checking PnL are those where future_return is not NaN
# This effectively drops the last row for computation purposes later
print(df_backtest[['Adj Close', 'future_return']].head())

            Adj Close  future_return
Date                                
1980-12-12   0.101073      -0.052170
1980-12-15   0.095800      -0.073403
1980-12-16   0.088768       0.024750
1980-12-17   0.090965       0.029000
1980-12-18   0.093603       0.061024


In [40]:
# 2. Create random signal
np.random.seed(42) # For reproducibility
# Use randint to match audit preference
signals = np.random.randint(0, 2, len(df_backtest))
df_backtest['long_only_signal'] = signals

print(df_backtest['long_only_signal'].head())

Date
1980-12-12    0
1980-12-15    1
1980-12-16    0
1980-12-17    0
1980-12-18    0
Name: long_only_signal, dtype: int64


In [41]:
# 3. Backtest signal
# If Signal=1, PnL = future_return * 1$
# If Signal=0, PnL = 0
df_backtest['strategy_pnl'] = df_backtest['long_only_signal'] * df_backtest['future_return']

print(df_backtest[['long_only_signal', 'future_return', 'strategy_pnl']].head())

            long_only_signal  future_return  strategy_pnl
Date                                                     
1980-12-12                 0      -0.052170     -0.000000
1980-12-15                 1      -0.073403     -0.073403
1980-12-16                 0       0.024750      0.000000
1980-12-17                 0       0.029000      0.000000
1980-12-18                 0       0.061024      0.000000


In [42]:
# 4. Compute strategy return
# Total Invested is the sum of days where we invested (signal=1)
# Only count invested on days where PnL is possible (i.e. future_return is not NaN)
# This is a subtle point. If we invest on the last day, we can't sell tomorrow.
# So we should drop the last day before summing invested.

valid_mask = df_backtest['future_return'].notna()
df_valid = df_backtest[valid_mask].copy()

total_invested_strategy = df_valid['long_only_signal'].sum()
total_pnl_strategy = df_valid['strategy_pnl'].sum()

strategy_return = total_pnl_strategy / total_invested_strategy

print(f"Total Invested: ${total_invested_strategy}")
print(f"Total PnL: ${total_pnl_strategy:.2f}")
print(f"Strategy Return: {strategy_return:.2%}")

Total Invested: $5045
Total PnL: $6.83
Strategy Return: 0.14%


In [43]:
# 5. Always Buy Strategy
# We update main DF first
df_backtest['always_buy_signal'] = 1
df_backtest['always_buy_pnl'] = df_backtest['always_buy_signal'] * df_backtest['future_return']

# Recalculate stats using the valid mask on the main DF to ensure we access the new column
total_invested_always = len(df_backtest[valid_mask])
total_pnl_always = df_backtest.loc[valid_mask, 'always_buy_pnl'].sum()
always_buy_return = total_pnl_always / total_invested_always

print(f"Total Invested (Always Buy): ${total_invested_always}")
print(f"Total PnL (Always Buy): ${total_pnl_always:.2f}")
print(f"Always Buy Return: {always_buy_return:.2%}")

Total Invested (Always Buy): $10116
Total PnL (Always Buy): $11.43
Always Buy Return: 0.11%


In [44]:
# 6. Plot Daily PnL
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_backtest.index, y=df_backtest['strategy_pnl'], mode='lines', name='Random Strategy PnL'))
fig.add_trace(go.Scatter(x=df_backtest.index, y=df_backtest['always_buy_pnl'], mode='lines', name='Always Buy PnL'))

fig.update_layout(title='Daily PnL: Random Strategy vs Always Buy', yaxis_title='PnL ($)')
fig.show()